# Deng 2025 Long-Range RIP: Eq. (1) Hamiltonian

This notebook builds the reduced Hamiltonian used in Deng et al., "Long-Range ZZ Interaction via Resonator-Induced Phase in Superconducting Qubits".

The focus here is the static Eq. (1) model: two qubits, two side driving resonators, and one central long-distance resonator mode. The drive terms used later for RIP dynamics are not included in Eq. (1); they enter the rotating-frame Hamiltonian in the supplemental material Eq. (S8).


## Model and units

We use the subsystem order

```text
(ql, qr, rl, rc, rr)
```

where `ql` and `qr` are two-level qubits, and `rl`, `rc`, `rr` are truncated harmonic resonators.

All parameters below are angular frequencies in `rad/ns`. Numerically, `2pi * 1 GHz = 2pi rad/ns`, and `1 MHz = 1e-3 GHz`.


In [1]:
using LinearAlgebra
using QuantumToolbox


In [2]:
const MHz = 1e-3
const two_pi = 2pi

Nq = 2
Nres = 5

params = (
    omega_l = two_pi * 5.000,
    omega_r = two_pi * 4.800,
    nu_l = two_pi * 6.9239,
    nu_c = two_pi * 6.9200,
    nu_r = two_pi * 6.9168,
    chi_ll = two_pi * 20.12 * MHz,
    chi_rr = two_pi * 20.22 * MHz,
    g_lc = two_pi * 100.0 * MHz,
    g_rc = two_pi * 100.0 * MHz,
)


(omega_l = 31.41592653589793, omega_r = 30.159289474462014, nu_l = 43.504146748380734, nu_c = 43.47964232568274, nu_r = 43.45953613269976, chi_ll = 0.12641768838045328, chi_rr = 0.12704600691117124, g_lc = 0.6283185307179586, g_rc = 0.6283185307179586)

## Eq. (1) as tensor-product operators

The implemented Hamiltonian is

```text
H = omega_l n_ql + omega_r n_qr
  + nu_l n_l + nu_c n_c + nu_r n_r
  + chi_ll n_l n_ql + chi_rr n_r n_qr
  + g_lc (a_l a_c' + a_l' a_c)
  + g_rc (a_r a_c' + a_r' a_c)
```

This is Eq. (1) specialized to local dispersive shifts between each qubit and its adjacent driving resonator.


In [3]:
function op_on(op, target::Symbol, sites, dims)
    local_ops = ntuple(i -> sites[i] == target ? op : qeye(dims[i]), length(sites))
    return tensor(local_ops...)
end

function excited_projector()
    return QuantumObject(ComplexF64[0 0; 0 1])
end

function build_deng_eq1_hamiltonian(params; Nq::Int=2, Nres::Int=5)
    sites = (:ql, :qr, :rl, :rc, :rr)
    dims = (Nq, Nq, Nres, Nres, Nres)

    n_exc = excited_projector()
    a = destroy(Nres)

    n_ql = op_on(n_exc, :ql, sites, dims)
    n_qr = op_on(n_exc, :qr, sites, dims)

    a_l = op_on(a, :rl, sites, dims)
    a_c = op_on(a, :rc, sites, dims)
    a_r = op_on(a, :rr, sites, dims)

    n_l = a_l' * a_l
    n_c = a_c' * a_c
    n_r = a_r' * a_r

    identity_full = tensor(ntuple(i -> qeye(dims[i]), length(dims))...)
    H = 0.0 * identity_full

    H += params.omega_l * n_ql + params.omega_r * n_qr
    H += params.nu_l * n_l + params.nu_c * n_c + params.nu_r * n_r
    H += params.chi_ll * n_l * n_ql + params.chi_rr * n_r * n_qr
    H += params.g_lc * (a_l * a_c' + a_l' * a_c)
    H += params.g_rc * (a_r * a_c' + a_r' * a_c)

    ops = (
        n_ql = n_ql,
        n_qr = n_qr,
        a_l = a_l,
        a_c = a_c,
        a_r = a_r,
        n_l = n_l,
        n_c = n_c,
        n_r = n_r,
    )

    return (H = H, ops = ops, sites = sites, dims = dims)
end

model = build_deng_eq1_hamiltonian(params; Nq=Nq, Nres=Nres)
H_eq1 = model.H



Quantum Object:   type=Operator()   dims=([2, 2, 5, 5, 5], [2, 2, 5, 5, 5])   size=(500, 500)   ishermitian=true
500×500 Matrix{ComplexF64}:
 0.0+0.0im       0.0+0.0im       0.0+0.0im  …      0.0+0.0im      0.0+0.0im
 0.0+0.0im   43.4595+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im   86.9191+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im  0.628319+0.0im       0.0+0.0im  …      0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im  0.888577+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im         0.0+0.0im      0.0+0.0im
    ⋮                                       ⋱                 
 0.0+0.0im       0.0+0.0im       0.

In [4]:
H_dense = Matrix(H_eq1.data)

(
    sites = model.sites,
    dims = model.dims,
    hilbert_dim = prod(model.dims),
    matrix_size = size(H_dense),
    hermitian = ishermitian(H_dense),
)


(sites = (:ql, :qr, :rl, :rc, :rr), dims = (2, 2, 5, 5, 5), hilbert_dim = 500, matrix_size = (500, 500), hermitian = true)

In [5]:
# Small static-spectrum check. Values are reported in GHz by dividing by 2pi.
evals = eigen(Hermitian(H_dense)).values
round.(evals[1:10] ./ two_pi; digits=6)


10-element Vector{Float64}:
  0.0
  4.8
  5.0
  6.778709
  6.92035
  7.061641
  9.8
 11.583571
 11.730438
 11.783081

## Next step

For the RIP gate simulation and `xi_ZZ`, add the Eq. (S8) drive terms and evolve either the full density matrix with `mesolve` or the coherent amplitudes from Eq. (S10)/(S19). This Eq. (1) object is the static reduced-system starting point.


## Static reduced Hamiltonian: Eq. (1)

Deng et al.의 reduced model은 두 qubit과 세 resonator mode를 다음 Hamiltonian으로 쓴다.

\[
H_0 =
\sum_{q=l,r} \tilde{\omega}_{01}^{(q)}\, \sigma_q^\dagger \sigma_q
+ \sum_{p=l,c,r} \tilde{\nu}_{p}\, b_p^\dagger b_p
+ \sum_{q=l,r} \chi^{(q,q)} b_q^\dagger b_q\, \sigma_q^\dagger \sigma_q
+ \sum_{p=l,r} g^{(p,c)}\left(b_p b_c^\dagger + b_p^\dagger b_c\right).
\]

여기서 \(q=l,r\)은 left/right qubit, \(p=l,c,r\)은 left/center/right resonator를 의미한다. 첫 두 항은 bare qubit/resonator energy이고, 세 번째 항은 각 qubit이 인접한 driving resonator의 frequency를 qubit state에 따라 shift하는 dispersive term이다. 마지막 항은 left/right driving resonator와 central long-distance resonator mode 사이의 hopping이다.

노트북의 `build_deng_eq1_hamiltonian`은 이 식을 subsystem order `(ql, qr, rl, rc, rr)`에 맞춰 tensor product operator로 만든다.


## Driven rotating-frame Hamiltonian: Eq. (S8)

`xi_ZZ`는 정적 \(H_0\)만으로 얻는 값이 아니라, 두 driving resonator에 microwave drive를 넣은 rotating-frame dynamics에서 나온다. 보충자료 Eq. (S8)의 형태는

\[
H_R(t) =
\sum_{p=l,c,r} \Delta_p b_p^\dagger b_p
+ \sum_{q=l,r} \chi^{(q,q)} b_q^\dagger b_q\, \sigma_q^\dagger \sigma_q
+ \sum_{p=l,r} g^{(p,c)}\left(b_p b_c^\dagger + b_p^\dagger b_c\right)
+ \frac{1}{2}\sum_{p=l,r}\left[\epsilon_p(t)e^{i\phi_p} b_p^\dagger + \epsilon_p^*(t)e^{-i\phi_p} b_p\right].
\]

\(\Delta_p = \nu_p - \omega_{d,p}\)는 drive frame에서 본 resonator detuning이다. 마지막 drive 항이 resonator coherent amplitude를 만들고, 그 amplitude가 qubit state에 따라 다르게 움직이면서 controlled phase를 누적한다.

현재 `rip` NamedTuple의 `epsilon_l`, `epsilon_r`는 이미 phase를 포함한 complex amplitude로 둔다. 예를 들어 antiphase IQ drive는 `epsilon_r = -epsilon_l`처럼 쓴다.


## Qubit-state-dependent resonator amplitude

Computational state를 \(|jk\rangle\), \(j,k\in\{0,1\}\)로 고정하면 qubit operator는 c-number처럼 작용한다. 따라서 세 resonator의 coherent amplitude

\[
\vec{\alpha}_{jk}(t) =
\begin{pmatrix}
\alpha_{jk}^{(l)}(t) \\
\alpha_{jk}^{(c)}(t) \\
\alpha_{jk}^{(r)}(t)
\end{pmatrix}
\]

는 선형 방정식을 따른다.

\[
\frac{d}{dt}\vec{\alpha}_{jk}(t) = G_{jk}\vec{\alpha}_{jk}(t) + \vec{E}(t),
\]

with the sign convention used in the code,

\[
G_{jk} =
\begin{pmatrix}
-\left(\frac{\kappa_l}{2}+i(\Delta_l-j\chi_l)\right) & -i g_{lc} & 0 \\
-i g_{lc} & -\left(\frac{\kappa_c}{2}+i\Delta_c\right) & -i g_{rc} \\
0 & -i g_{rc} & -\left(\frac{\kappa_r}{2}+i(\Delta_r-k\chi_r)\right)
\end{pmatrix},
\]

and

\[
\vec{E}(t) = -\frac{i}{2}
\begin{pmatrix}
\epsilon_l(t) \\
0 \\
\epsilon_r(t)
\end{pmatrix}.
\]

For a constant-amplitude drive, the steady-state solution is

\[
\vec{\alpha}_{jk}^{\,\mathrm{ss}} = -G_{jk}^{-1}\vec{E}.
\]

The code below starts with this steady-state approximation because it is cheap and directly shows the detuning dependence of the RIP interaction.


## Accumulated phase and `xi_ZZ`

The resonator-induced phase between two qubit-state branches \(|jk\rangle\) and \(|lm\rangle\) evolves as

\[
\dot{\mu}_{jk,lm}(t) =
-\sum_{p=l,c,r}
\left(\tilde{\chi}_{jk}^{(p)}-\tilde{\chi}_{lm}^{(p)}\right)
\alpha_{jk}^{(p)}(t)\,\alpha_{lm}^{(p)}(t)^*.
\]

The controlled two-qubit phase is the phase combination

\[
\theta_{ZZ}(t) =
\mu_{00,00}(t)+\mu_{11,00}(t)-\mu_{10,00}(t)-\mu_{01,00}(t).
\]

The ZZ interaction strength is the slope of this phase,

\[
\xi_{ZZ} = \frac{d\theta_{ZZ}}{dt}.
\]

In the notebook units, angular frequencies are in `rad/ns`, so the plotted MHz value is

\[
\frac{\xi_{ZZ}}{2\pi}\,[\mathrm{MHz}]
= \frac{\mathrm{Re}\left[\dot{\theta}_{ZZ}\right]}{2\pi\times 10^{-3}}.
\]

The corresponding ideal CZ time is

\[
t_{CZ} = \frac{\pi}{\left|\mathrm{Re}\left[\xi_{ZZ}\right]\right|}.
\]

The imaginary part of \(\dot{\theta}_{ZZ}\) tracks photon-loss-induced dephasing when finite \(\kappa_p\) is included.


## Getting `xi_ZZ` from driven resonator amplitudes

The code below implements the equations above for the four computational states and evaluates the steady-state phase slope. This is the low-cost numerical route before running a larger full Hilbert-space `mesolve` simulation.

The sign convention below matches the supplemental Eq. (S20): when a qubit is excited, the adjacent resonator detuning is shifted as `Delta - chi`. The reported gate rate usually uses `abs(xi_ZZ)`.


In [6]:
rip = (
    Delta_l = two_pi * 80.0 * MHz,
    Delta_c = two_pi * 80.0 * MHz,
    Delta_r = two_pi * 80.0 * MHz,
    epsilon_l = ComplexF64(two_pi * 200.0 * MHz),
    epsilon_r = ComplexF64(-two_pi * 200.0 * MHz),
    kappa_l = 0.0,
    kappa_c = 0.0,
    kappa_r = 0.0,
    chi_l = params.chi_ll,
    chi_r = params.chi_rr,
    g_lc = params.g_lc,
    g_rc = params.g_rc,
)


(Delta_l = 0.5026548245743669, Delta_c = 0.5026548245743669, Delta_r = 0.5026548245743669, epsilon_l = 1.2566370614359172 + 0.0im, epsilon_r = -1.2566370614359172 + 0.0im, kappa_l = 0.0, kappa_c = 0.0, kappa_r = 0.0, chi_l = 0.12641768838045328, chi_r = 0.12704600691117124, g_lc = 0.6283185307179586, g_rc = 0.6283185307179586)

In [7]:
function amplitude_matrix(label::Tuple{Int, Int}, rip)
    left_excited, right_excited = label

    Delta_l_eff = rip.Delta_l - left_excited * rip.chi_l
    Delta_c_eff = rip.Delta_c
    Delta_r_eff = rip.Delta_r - right_excited * rip.chi_r

    return ComplexF64[
        -(rip.kappa_l / 2 + 1im * Delta_l_eff)  -1im * rip.g_lc                         0;
        -1im * rip.g_lc                         -(rip.kappa_c / 2 + 1im * Delta_c_eff)  -1im * rip.g_rc;
        0                                       -1im * rip.g_rc                         -(rip.kappa_r / 2 + 1im * Delta_r_eff)
    ]
end

drive_vector(rip) = ComplexF64[-1im * rip.epsilon_l / 2, 0, -1im * rip.epsilon_r / 2]

function steady_amplitude(label::Tuple{Int, Int}, rip)
    return -(amplitude_matrix(label, rip) \ drive_vector(rip))
end

function dispersive_shift_vector(label::Tuple{Int, Int}, rip)
    left_excited, right_excited = label
    return ComplexF64[left_excited * rip.chi_l, 0, right_excited * rip.chi_r]
end

function mu_dot(alpha_jk, alpha_lm, jk::Tuple{Int, Int}, lm::Tuple{Int, Int}, rip)
    chi_diff = dispersive_shift_vector(jk, rip) .- dispersive_shift_vector(lm, rip)
    return -sum(chi_diff .* alpha_jk .* conj.(alpha_lm))
end

function theta_dot_zz_steady(rip)
    labels = ((0, 0), (1, 0), (0, 1), (1, 1))
    alpha = Dict(label => steady_amplitude(label, rip) for label in labels)
    ref = (0, 0)

    return mu_dot(alpha[(0, 0)], alpha[ref], (0, 0), ref, rip) +
           mu_dot(alpha[(1, 1)], alpha[ref], (1, 1), ref, rip) -
           mu_dot(alpha[(1, 0)], alpha[ref], (1, 0), ref, rip) -
           mu_dot(alpha[(0, 1)], alpha[ref], (0, 1), ref, rip)
end

function xi_zz_summary(rip)
    theta_dot = theta_dot_zz_steady(rip)
    xi_MHz = real(theta_dot) / (two_pi * MHz)
    decay_MHz = imag(theta_dot) / (two_pi * MHz)
    t_CZ_ns = pi / abs(real(theta_dot))

    return (
        theta_dot_rad_per_ns = theta_dot,
        xi_ZZ_over_2pi_MHz = xi_MHz,
        Im_theta_dot_over_2pi_MHz = decay_MHz,
        CZ_time_ns = t_CZ_ns,
    )
end


xi_zz_summary (generic function with 1 method)

In [8]:
xi_zz_summary(rip)


(theta_dot_rad_per_ns = -0.1051908377160056 + 0.0im, xi_ZZ_over_2pi_MHz = -16.74164179047967, Im_theta_dot_over_2pi_MHz = 0.0, CZ_time_ns = 29.865649155409052)

## Detuning sweep

Sweeping `Delta` reproduces the basic Fig. 3(a) workflow: rebuild `rip` for each detuning, compute the steady-state phase slope, and plot or compare `xi_ZZ / 2pi` in MHz.


In [9]:
function with_detuning(rip, Delta_MHz)
    Delta = two_pi * Delta_MHz * MHz
    return merge(rip, (Delta_l = Delta, Delta_c = Delta, Delta_r = Delta))
end

Delta_MHz_values = collect(40.0:10.0:200.0)
xi_sweep_MHz = [xi_zz_summary(with_detuning(rip, Delta_MHz)).xi_ZZ_over_2pi_MHz
                for Delta_MHz in Delta_MHz_values]

collect(zip(Delta_MHz_values, round.(xi_sweep_MHz; digits=4)))


17-element Vector{Tuple{Float64, Float64}}:
 (40.0, -181.0512)
 (50.0, -75.3776)
 (60.0, -39.825)
 (70.0, -24.4479)
 (80.0, -16.7416)
 (90.0, -12.5369)
 (100.0, -10.1923)
 (110.0, -9.0352)
 (120.0, -8.928)
 (130.0, -10.5108)
 (140.0, -19.4287)
 (150.0, 34.0055)
 (160.0, 6.2558)
 (170.0, 2.8225)
 (180.0, 1.5887)
 (190.0, 0.9957)
 (200.0, 0.6664)

For a full QuantumToolbox master-equation simulation, use the same rotating-frame terms as Eq. (S8), add the two time-dependent drive operators, evolve the two Ramsey initial states `|0,+,0,0,0>` and `|1,+,0,0,0>`, then fit the slope of the relative target-qubit phase. The coherent-amplitude calculation above is the low-cost numerical route used before committing to the much larger Hilbert-space simulation.
